# Comparação de eficiência de CNNs — ISIC 2019

Driver para rodar o estudo no Google Colab.

**Antes de começar:** `Runtime ▸ Change runtime type ▸ GPU`.

Rode as células na ordem. `results/`, `checkpoints/` e `figs/` são gravados
no seu Google Drive, então o progresso sobrevive a quedas de sessão.
O treino é **resumível**: se a sessão cair, re-execute a mesma célula de treino.

## 1. Setup — clona o repositório e instala dependências

Faz um clone limpo a cada execução (o código é pequeno; `results/`,
`checkpoints/` e `figs/` ficam no Drive, então nada se perde).

Pode re-executar à vontade. **Rode sempre a célula 2 depois desta** — o clone
sobrescreve o `config.yaml`, e é a célula 2 que aponta as saídas para o Drive.

In [ ]:
import os

REPO_URL = "https://github.com/pedruck/skin-cnn-efficiency.git"
REPO_DIR = "/content/skin-cnn-efficiency"

# Sair do repo ANTES de apaga-lo. Ao re-executar esta celula o CWD ja e REPO_DIR
# (por causa do chdir la embaixo); apagar o proprio CWD mata o getcwd e o git
# clone falha com "Unable to read current working directory".
os.chdir("/content")

!rm -rf "{REPO_DIR}" && git clone "{REPO_URL}" "{REPO_DIR}"
assert os.path.isfile(f"{REPO_DIR}/config.yaml"), "clone falhou — veja o erro acima"
os.chdir(REPO_DIR)
!pip install -q thop kagglehub pyyaml nvidia-ml-py
print("OK — repo em", REPO_DIR)
print("Rode a celula 2 em seguida: o clone restaurou o config.yaml original,")
print("entao sem ela as saidas NAO vao para o Drive e o resume nao acha nada.")

## 2. Google Drive para as saídas persistentes

As saídas (`results/`, `checkpoints/`, `figs/`) são gravadas direto numa pasta do
seu Drive — os caminhos são escritos no config. Assim o progresso sobrevive a
quedas de sessão.

Cada config declara um `experiment`, e as saídas vão para uma subpasta com esse
nome (`4classes/`, `8classes/`). Os dois cenários nunca se sobrescrevem.

In [ ]:
import yaml
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT = '/content/drive/MyDrive/skin-cnn-efficiency'

for path in ('config.yaml', 'config_8classes.yaml'):
    cfg = yaml.safe_load(open(path))          # cru: preserva a chave 'base:'
    root = f"{DRIVE_ROOT}/{cfg.get('experiment', 'default')}"
    for sub in ('results', 'checkpoints', 'figs'):
        os.makedirs(f'{root}/{sub}', exist_ok=True)
    cfg['paths'] = {'results_dir':     f'{root}/results',
                    'checkpoints_dir': f'{root}/checkpoints',
                    'figures_dir':     f'{root}/figs'}
    with open(path, 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
    print(f'{path:24s} -> {root}')

## 3. Dataset ISIC 2019 (download via kagglehub)

Re-baixa a cada sessão (poucos GB). Se pedir credencial, faça upload do
`kaggle.json` (Kaggle ▸ Account ▸ Create New Token) em `/root/.config/kaggle/`.

In [ ]:
import kagglehub
DATA = kagglehub.dataset_download('salviohexia/isic-2019-skin-lesion-images-for-classification')
print('DATA =', DATA)

## 3b. Cache de imagens 256x256 (recomendado)

O treino reduz toda imagem a 256x256 de qualquer forma — o modelo nunca ve
mais que isso. Refazer esse resize a cada epoca, em ~25 mil JPEGs de ~1024x768,
faz do DataLoader virar o gargalo: a GPU espera o disco e `epoch_time_s` passa a
medir o **armazenamento** em vez da arquitetura — o que invalidaria a comparacao
de custo de treino do artigo.

Converte uma vez (~5-10 min) e aponta `DATA` para o cache. E resumivel.

**Pode pular esta celula:** o treino usa o dataset original e funciona igual.
O cache fica em `/content` (efemero) de proposito — sao 25 mil arquivos pequenos,
gravar isso no Drive seria lento e ele e reconstruivel em minutos.

In [ ]:
# 1) as imagens de origem sao grandes o bastante para o cache valer a pena?
!python -m src.prepare_cache --data "$DATA" --out /content/isic256 --inspect

# 2) converte e passa a usar o cache dali em diante
#    (se o passo 1 mostrar dimensoes <= 256, comente as duas linhas abaixo)
!python -m src.prepare_cache --data "$DATA" --out /content/isic256
DATA = '/content/isic256'

print('DATA =', DATA)

## 4. Treino

**Uma arquitetura por célula.** ~25–55 min cada numa T4 (ResNet-50 é a mais lenta).
Se a sessão cair, é só re-executar a célula: retoma da última época salva.

Ajuste as épocas em `config.yaml` (padrão: 20) ou com `--epochs`.

O padrão é o cenário de **4 classes** (MEL, NV, BCC, BKL). Para o de 8,
acrescente `--config config_8classes.yaml` em cada comando desta seção,
no benchmark e nas figuras.

In [ ]:
!python -m src.train --model resnet50 --data "$DATA"

In [ ]:
!python -m src.train --model resnet18 --data "$DATA"

In [ ]:
!python -m src.train --model mobilenet_v2 --data "$DATA"

In [ ]:
!python -m src.train --model mobilenet_v3_large --data "$DATA"

## 5. Benchmark — avaliação no teste + métricas de eficiência

Gera `results/performance.csv`, `efficiency.csv`, `summary.csv`,
`environment.json` e `test_predictions_<modelo>.npz`.

In [ ]:
!python -m src.benchmark --data "$DATA"

## 6. Figuras

In [ ]:
!python -m src.plots --data "$DATA"

import glob, yaml
from IPython.display import Image, display
figs_dir = yaml.safe_load(open('config.yaml'))['paths']['figures_dir']
for f in sorted(glob.glob(f'{figs_dir}/*.png')):
    print(f); display(Image(f))

## 7. Baixar artefatos

Também já estão no seu Drive em `MyDrive/skin-cnn-efficiency/`.

In [ ]:
import os, shutil, yaml
p = yaml.safe_load(open('config.yaml'))['paths']
os.makedirs('/content/artefatos', exist_ok=True)
for key in ('results_dir', 'figures_dir'):
    if os.path.isdir(p[key]):
        shutil.copytree(p[key], f"/content/artefatos/{os.path.basename(p[key])}",
                        dirs_exist_ok=True)
shutil.make_archive('/content/artefatos', 'zip', '/content/artefatos')
from google.colab import files
files.download('/content/artefatos.zip')